In [1]:
import pandas as pd 
import re
pathmoe = '/hdd3/sonia/be_great/ckpts/moe/dgpt2/adult-allcol/jul21/samples'
pathoh = '/hdd3/sonia/be_great/ckpts/dgpt2/adult-allcol/samples'
pathreal = './adult.csv'
pathgreatdgpt2 = '/hdd3/sonia/be_great/ckpts/dgpt2-great'
pathgreatmoe = '/home/sonia/be_great/ckpts/great/adult/moedgpt2-aug11/samples'
path = pathgreatmoe

age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income

### From text

In [2]:
raws = []
with open(path+'.txt', 'r') as f:
    for raw in f.readlines():
        raws.append(re.sub('is\?', 'is ?', raw))

In [3]:
l = raws[0]
problem = 0
def parse_line(l):
    cols = l.split('.<EOS>')[:-1] # remove newline at end
    words = [c.split(' ') for c in cols] #'name', 'is', 'value'
    if len(words) == 15:
        return {c[0]:c[2] for c in words}
    else:
        problem += 1
        return {}

line_dicts = [parse_line(l) for l in raws]
df = pd.DataFrame.from_records(line_dicts)
print(problem, 'problem lines')
df

0 problem lines


,income,class,gain-capital,sex,relationship,years-education,loss-capital,hours-per-week,occupation,marital-status,financial-weight,native-country,race,education,age
0,<=50K,Private,0,Female,Unmarried,10,0,35,Adm-clerical,Divorced,151055,United-States,White,Some-college,33
1,>50K,Private,0,Male,Husband,14,0,40,Prof-specialty,Married-civ-spouse,264767,United-States,White,Masters,45
2,>50K,Private,0,Male,Husband,10,0,40,Sales,Married-civ-spouse,163521,United-States,White,Some-college,40
3,<=50K,Private,0,Female,Unmarried,12,0,40,Adm-clerical,Never-married,235950,United-States,Asian-Pac-Islander,Assoc-acdm,22
4,>50K,Self-emp-not-inc,0,Male,Husband,10,0,60,Exec-managerial,Married-civ-spouse,253770,United-States,White,Some-college,48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,>50K,Private,0,Male,Husband,13,0,40,Exec-managerial,Married-civ-spouse,262558,United-States,White,Bachelors,47
9996,>50K,Private,0,Male,Husband,10,0,50,Handlers-cleaners,Married-civ-spouse,210311,United-States,White,Some-college,41
9997,<=50K,Private,0,Male,Husband,13,0,50,Adm-clerical,Married-civ-spouse,122397,United-States,White,Bachelors,49
9998,<=50K,Private,0,Male,Husband,10,0,40,Sales,Married-civ-spouse,362384,United-States,White,Some-college,62


In [4]:
cols = ['age', 'class', 'financial-weight', 'education', 'years-education', 'marital-status', 'occupation', 'relationship', 
        'race', 'sex', 'gain-capital', 'loss-capital', 'hours-per-week', 'native-country', 'income']
colsgreatreal = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 
        'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
colsdict = {g:r for (g,r) in zip(cols, colsgreatreal)}
df.columns = [colsdict[c] for c in df.columns]
df = df[colsgreatreal]
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,33,Private,151055,Some-college,10,Divorced,Adm-clerical,Unmarried,White,Female,0,0,35,United-States,<=50K
1,45,Private,264767,Masters,14,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,>50K
2,40,Private,163521,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,>50K
3,22,Private,235950,Assoc-acdm,12,Never-married,Adm-clerical,Unmarried,Asian-Pac-Islander,Female,0,0,40,United-States,<=50K
4,48,Self-emp-not-inc,253770,Some-college,10,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,60,United-States,>50K


### From csv

In [5]:
# df=pd.read_csv(pathgreatdgpt2+'.csv')
# df['income'] = df['income'].apply(lambda x: x.strip())
# df.head()

### Match with real values

In [6]:
real = pd.read_csv(pathreal)
colsgreatreal = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 
        'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
real.columns = colsgreatreal
ords = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']
ordvals = {col:set(real[col].unique()) for col in ords}
for col in ordvals:
    ordvals[col] = [val.strip() for val in ordvals[col]]
# ordvals

In [7]:
for col in ordvals:
    df = df[df[col].isin(ordvals[col])]
    print(col, len(df))
# df[[df[col].isin(ordvals[col]) for col in ordvals]]
df
# df=df[df['workclass'].isin(ordvals['workclass']) & df['income'].isin(ordvals['income'])]
# df.head()

workclass 9999
education 9998
marital-status 9998
occupation 9997
relationship 9997
race 9997
sex 9997
native-country 9997
income 9997


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,33,Private,151055,Some-college,10,Divorced,Adm-clerical,Unmarried,White,Female,0,0,35,United-States,<=50K
1,45,Private,264767,Masters,14,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,>50K
2,40,Private,163521,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,>50K
3,22,Private,235950,Assoc-acdm,12,Never-married,Adm-clerical,Unmarried,Asian-Pac-Islander,Female,0,0,40,United-States,<=50K
4,48,Self-emp-not-inc,253770,Some-college,10,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,60,United-States,>50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,47,Private,262558,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,>50K
9996,41,Private,210311,Some-college,10,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,50,United-States,>50K
9997,49,Private,122397,Bachelors,13,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,50,United-States,<=50K
9998,62,Private,362384,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,<=50K


In [8]:
df.to_csv(path+'clean.csv', index=False)